# TAMER OCR v2.2 — Unified Execution Orchestrator

**Production Pipeline Strategy:**
1. **Environment Purge:** Clean workspace to prevent cache contamination.
2. **Authentication:** Secure setup for HuggingFace and Kaggle.
3. **Dependencies:** Install fixed requirements (including `psutil`).
4. **Execution:** Run the strict Preprocessing -> Train -> Sync pipeline.
5. **Dynamic Scaling:** Optimizer steps are calculated automatically based on data size.

In [ ]:
import os
import shutil
import gc

print("🧹 Cleaning workspace for a fresh start...")
IS_KAGGLE = os.path.exists('/kaggle')
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
os.chdir(WORK_DIR)

directories = ['data', 'outputs', 'checkpoints', 'logs', 'AI_PROJECT_TAMER_Complete']

for d in directories:
    path = os.path.join(WORK_DIR, d)
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)
        print(f"  - Removed: {path}")

gc.collect()
print("✅ Workspace is clean.")

In [ ]:
import getpass
import os

print("🔑 Configure credentials:")

# Hugging Face Auth
hf_token = os.getenv('HF_TOKEN') or getpass.getpass('HuggingFace Token (Write Access): ')
os.environ['HF_TOKEN'] = hf_token

# Kaggle Auth
kaggle_user = getpass.getpass('Kaggle Username: ')
kaggle_key = getpass.getpass('Kaggle API Key: ')
os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = kaggle_user, kaggle_key

kaggle_conf = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_conf, exist_ok=True)
with open(os.path.join(kaggle_conf, 'kaggle.json'), 'w') as f:
    f.write(f'{{"username":"{kaggle_user}","key":"{kaggle_key}"}}')
os.chmod(os.path.join(kaggle_conf, 'kaggle.json'), 0o600)

print("✅ Authentication configured.")

In [ ]:
print("📦 Installing fixed dependencies...")
# Added psutil explicitly to prevent preprocessor crashes
!pip install -q timm albumentations datasets huggingface_hub requests tqdm psutil opencv-python-headless

print("\n📥 Cloning production codebase...")
# Replace with your actual repository URL
REPO_URL = "https://github.com/Vtheonly/AI_PROJECT_TAMER_Complete"
!git clone {REPO_URL}

print("✅ System ready.")

In [ ]:
import sys
from huggingface_hub import HfApi

WORK_DIR = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
REPO_PATH = os.path.join(WORK_DIR, 'AI_PROJECT_TAMER_Complete')
if REPO_PATH not in sys.path: sys.path.insert(0, REPO_PATH)

from tamer_ocr.config import Config
config = Config()

# Environment-specific path mapping
config.data_dir = os.path.join(WORK_DIR, 'data')
config.output_dir = os.path.join(WORK_DIR, 'outputs')
config.checkpoint_dir = os.path.join(WORK_DIR, 'checkpoints')
config.log_dir = os.path.join(WORK_DIR, 'logs')

for d in [config.data_dir, config.output_dir, config.checkpoint_dir, config.log_dir]: 
    os.makedirs(d, exist_ok=True)

# Hugging Face Repository Setup
hf_api = HfApi(token=os.environ['HF_TOKEN'])
hf_user = hf_api.whoami()['name']
config.hf_token = os.environ['HF_TOKEN']
config.hf_repo_id = f"{hf_user}/tamer-ocr-model"
config.hf_dataset_repo_id = f"{hf_user}/tamer-preprocessed"

print(f"✅ Configured for HF User: {hf_user}")
print(f"✅ Model Repo: {config.hf_repo_id}")

In [ ]:
from tamer_ocr.core.trainer import Trainer

print("🚀 Launching TAMER Core Training...")
print("Note: Preprocessing will run first. Training starts automatically after sync.")

trainer = Trainer(config)
try:
    # This runs the full pipeline: Preprocess -> Build -> Resume -> Train
    trainer.run()
except Exception as e:
    print(f"\n❌ Pipeline failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
from tamer_ocr.utils.checkpoint import push_checkpoint_to_hf

print("📤 Performing final artifact synchronization...")
best_pt = os.path.join(config.checkpoint_dir, 'best.pt')

if os.path.exists(best_pt):
    push_checkpoint_to_hf(best_pt, config, epoch=0, is_best=True)
    print("✅ Best model successfully synced to HuggingFace Hub.")
else:
    print("⚠️ No best.pt found. Check logs for training status.")